# Cohen Statins × BiomedBERT — Multi-Seed Sensitivity for H-Pub2

Runs the two H-Pub2-critical text modes (`title_abstract_mesh` and `auto_mesh`) on the full Cohen Statins corpus with BiomedBERT on T4 GPU, across five random seeds, to characterise the seed-sensitivity of the H-Pub2 falsification.

**Why this run exists.** The H-Pub2 falsification documented in `paper_draft_v1.md` and `cohen_benchmark_bert_extension.md` §6 rests on seed=42 only. Paper Limitations §6 paragraph 3 acknowledges this. Five-seed runs at the same hyperparameters address that limitation directly and are computationally cheap (~5 hours T4 GPU).

**Configuration:**
- Topic: Statins (2,744 abstracts, 152 included)
- Modes: `title_abstract_mesh` (expert MeSH) and `auto_mesh` (mechanical MeSH) — the two modes that define the H-Pub2 gap
- Seeds: 42, 7, 13, 21, 31
- 5-fold stratified CV, 3 epochs, lr=2e-5, batch=16, class_weight=balanced, fp16
- Per-fold raw values saved to `.json` sibling files for downstream pooling

**Optional baseline cells (4a, 4b)** are retained for reproducibility but are not part of the multi-seed run. They use seed=42 only since the baseline modes do not enter the H-Pub2 verdict.

**Expected runtime:** ~5 hours total on T4 (5 seeds × 2 modes × ~30 min). The cell is idempotent — if a seed's output file already exists, that seed is skipped, so disconnects can be recovered by rerunning the cell.

## 1. Setup — mount, clone, copy files, install dependenciesRun this once at the start of every Colab session.

In [ ]:
# Colab preflight check for BERT pipeline run
# Paste this into a single Colab cell and run it.
# It performs all checks without running any actual training.
# Expected runtime: under 60 seconds.

import os
import sys
import subprocess
import zipfile
import shutil
from pathlib import Path

PASS = "[PASS]"
FAIL = "[FAIL]"
WARN = "[WARN]"
INFO = "[INFO]"

print("=" * 70)
print("BERT Pipeline Preflight Check")
print("=" * 70)
errors = []
warnings = []


# 1. Runtime check
print("\n--- 1. Runtime environment ---")
print(f"{INFO} Python: {sys.version.split()[0]}")

try:
    import torch
    print(f"{INFO} torch: {torch.__version__}")
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"{PASS} GPU detected: {gpu_name} ({gpu_mem:.1f} GB)")
    else:
        print(f"{FAIL} No GPU available. Switch runtime: Runtime > Change runtime type > T4 GPU")
        errors.append("No GPU")
except ImportError:
    print(f"{INFO} torch not yet installed (Colab will install in main notebook)")


# 2. Mount Google Drive
print("\n--- 2. Google Drive ---")
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print(f"{PASS} Drive mounted at /content/drive")
except Exception as exc:
    print(f"{FAIL} Drive mount failed: {exc}")
    errors.append("Drive mount")


# 3. Verify cohen_bert_run folder and required files
print("\n--- 3. Drive contents ---")
DRIVE_ROOT = Path("/content/drive/MyDrive/cohen_bert_run")
required_files = ["cohen_data.zip", "bert_models.py", "cohen_bert_pipeline.py"]

if not DRIVE_ROOT.exists():
    print(f"{FAIL} Folder not found: {DRIVE_ROOT}")
    print(f"       Make sure 'cohen_bert_run' is at MyDrive root, exact spelling.")
    errors.append("Drive folder missing")
else:
    print(f"{PASS} Folder found: {DRIVE_ROOT}")
    for fname in required_files:
        fpath = DRIVE_ROOT / fname
        if fpath.exists():
            size_kb = fpath.stat().st_size / 1024
            print(f"{PASS} {fname} ({size_kb:.1f} KB)")
        else:
            print(f"{FAIL} {fname} missing in Drive folder")
            errors.append(f"{fname} missing")


# 4. Verify zip contents
print("\n--- 4. Zip integrity ---")
zip_path = DRIVE_ROOT / "cohen_data.zip"
if zip_path.exists():
    try:
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
            n_files = len(names)
            json_count = sum(1 for n in names if n.endswith(".json"))
            tsv_present = any("epc-ir.clean.tsv" in n for n in names)

            print(f"{INFO} Files in zip: {n_files}")
            print(f"{INFO} JSON cache files: {json_count}")

            if tsv_present:
                print(f"{PASS} TSV file present in zip")
            else:
                print(f"{WARN} epc-ir.clean.tsv not found in zip — check zip contents")
                warnings.append("TSV may be missing from zip")

            if json_count < 3000:
                print(f"{WARN} Expected ~3,463 JSON cache files; got {json_count}")
                warnings.append("Fewer cache files than expected")
            else:
                print(f"{PASS} Cache file count reasonable")

            # Peek at first few entries
            print(f"{INFO} Sample paths in zip:")
            for n in names[:3]:
                print(f"       {n}")
    except zipfile.BadZipFile:
        print(f"{FAIL} cohen_data.zip is corrupted")
        errors.append("Bad zip")


# 5. Verify .py files look right (basic header check)
print("\n--- 5. Python file sanity ---")
for fname, expected_marker in [
    ("bert_models.py", "BiomedBertClassifier"),
    ("cohen_bert_pipeline.py", "run_bert_kfold"),
]:
    fpath = DRIVE_ROOT / fname
    if fpath.exists():
        content = fpath.read_text(encoding="utf-8", errors="ignore")
        if expected_marker in content:
            line_count = len(content.splitlines())
            print(f"{PASS} {fname} contains '{expected_marker}' ({line_count} lines)")
        else:
            print(f"{FAIL} {fname} does not contain '{expected_marker}' — wrong file?")
            errors.append(f"{fname} content unexpected")

    # Specific check for class weighting in bert_models.py
    if fname == "bert_models.py" and fpath.exists():
        if "class_weight" in content and "WeightedLossTrainer" in content:
            print(f"{PASS} bert_models.py has class weighting support")
        else:
            print(f"{FAIL} bert_models.py missing class weighting — using old version?")
            errors.append("Old bert_models.py without class weighting")

    # Specific check for Tee fix in cohen_bert_pipeline.py
    if fname == "cohen_bert_pipeline.py" and fpath.exists():
        if "isatty" in content and "__getattr__" in content:
            print(f"{PASS} cohen_bert_pipeline.py has Tee isatty fix")
        else:
            print(f"{WARN} cohen_bert_pipeline.py may lack Tee fix — file outputs may crash")
            warnings.append("Old cohen_bert_pipeline.py without Tee fix")


# 6. HuggingFace token
print("\n--- 6. HuggingFace token ---")
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token and len(token) > 10:
        print(f"{PASS} HF_TOKEN accessible from Colab Secrets (length {len(token)})")
        os.environ["HF_TOKEN"] = token
    else:
        print(f"{WARN} HF_TOKEN secret exists but looks empty/short")
        warnings.append("HF token may be invalid")
except Exception as exc:
    print(f"{WARN} HF_TOKEN not set up (will work anonymously with rate limits): {exc}")
    warnings.append("No HF token (downloads will be slower, may hit rate limits)")


# 7. GitHub repo accessibility (without cloning yet)
print("\n--- 7. GitHub repo ---")
REPO_URL = "https://github.com/SamInMotion/Medical-intervention-text-classification"
result = subprocess.run(
    ["git", "ls-remote", "--heads", REPO_URL],
    capture_output=True, text=True, timeout=30,
)
if result.returncode == 0:
    branches = [line.split()[-1] for line in result.stdout.strip().splitlines()]
    print(f"{PASS} Repo accessible. Branches: {', '.join(b.split('/')[-1] for b in branches)}")
else:
    print(f"{FAIL} Cannot reach repo: {result.stderr.strip()}")
    errors.append("GitHub repo not accessible")


# 8. Disk space
print("\n--- 8. Disk space ---")
stat = shutil.disk_usage("/content")
free_gb = stat.free / 1e9
total_gb = stat.total / 1e9
print(f"{INFO} /content disk: {free_gb:.1f} GB free of {total_gb:.1f} GB")
if free_gb < 5:
    print(f"{WARN} Less than 5 GB free — may run out during model download + cache extraction")
    warnings.append("Low disk space")
else:
    print(f"{PASS} Sufficient disk space")


# Summary
print("\n" + "=" * 70)
print("Preflight Summary")
print("=" * 70)
if not errors and not warnings:
    print(f"{PASS} All checks passed. Ready to run the main notebook.")
elif not errors:
    print(f"{PASS} All required checks passed.")
    print(f"{WARN} {len(warnings)} warning(s):")
    for w in warnings:
        print(f"      - {w}")
    print(f"\n      Warnings are not blockers. Ready to proceed.")
else:
    print(f"{FAIL} {len(errors)} blocker(s) found:")
    for e in errors:
        print(f"      - {e}")
    print(f"\n      Fix these before running the main notebook.")
    if warnings:
        print(f"\n{WARN} Also: {len(warnings)} warning(s):")
        for w in warnings:
            print(f"      - {w}")

BERT Pipeline Preflight Check

--- 1. Runtime environment ---
[INFO] Python: 3.12.13
[INFO] torch: 2.11.0+cu128
[PASS] GPU detected: Tesla T4 (15.6 GB)

--- 2. Google Drive ---
Mounted at /content/drive
[PASS] Drive mounted at /content/drive

--- 3. Drive contents ---
[PASS] Folder found: /content/drive/MyDrive/cohen_bert_run
[PASS] cohen_data.zip (3840.3 KB)
[PASS] bert_models.py (9.7 KB)
[PASS] cohen_bert_pipeline.py (17.4 KB)

--- 4. Zip integrity ---
[INFO] Files in zip: 3466
[INFO] JSON cache files: 3463
[PASS] TSV file present in zip
[PASS] Cache file count reasonable
[INFO] Sample paths in zip:
       data/cohen/
       data/cohen/cache/
       data/cohen/cache/10023901.json

--- 5. Python file sanity ---
[PASS] bert_models.py contains 'BiomedBertClassifier' (277 lines)
[PASS] bert_models.py has class weighting support
[PASS] cohen_bert_pipeline.py contains 'run_bert_kfold' (513 lines)
[PASS] cohen_bert_pipeline.py has Tee isatty fix

--- 6. HuggingFace token ---
[PASS] HF_TOKEN

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

# Mount Drive (idempotent — won't re-prompt if already mounted)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# Paths
DRIVE_ROOT = Path("/content/drive/MyDrive/cohen_bert_run")
WORK_DIR = Path("/content/Medical-intervention-text-classification")
OUTPUTS_DIR = DRIVE_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Clean previous workdir if it exists (idempotent)
if WORK_DIR.exists():
    print(f"Removing previous workdir: {WORK_DIR}")
    shutil.rmtree(WORK_DIR)

# Clone repo (shallow — we only need the latest)
print("Cloning repo...")
subprocess.run(
    ["git", "clone", "--depth=1",
     "https://github.com/SamInMotion/Medical-intervention-text-classification.git",
     str(WORK_DIR)],
    check=True, capture_output=True,
)

# Copy new .py files from Drive into src/, overwriting whatever was on GitHub
print("Copying BERT files from Drive into src/...")
shutil.copy(DRIVE_ROOT / "bert_models.py", WORK_DIR / "src" / "bert_models.py")
shutil.copy(DRIVE_ROOT / "cohen_bert_pipeline.py", WORK_DIR / "src" / "cohen_bert_pipeline.py")

# Unzip Cohen data into the repo's data/ directory
print("Extracting cohen_data.zip...")
subprocess.run(
    ["unzip", "-q", "-o", str(DRIVE_ROOT / "cohen_data.zip"), "-d", str(WORK_DIR)],
    check=True,
)

# Install dependencies (torch, sklearn already on Colab)
print("Installing dependencies...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers>=5.0", "accelerate>=1.1.0", "biopython", "nltk"],
    check=True,
)

# Set environment variables — HF token, silence telemetry/verbose logging
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set from Colab Secrets")
except Exception as e:
    print(f"HF_TOKEN not available: {e} (will work anonymously)")

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "warning"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Change to work dir so `python -m src.cohen_bert_pipeline` resolves
os.chdir(WORK_DIR)

# Verify GPU
import torch
assert torch.cuda.is_available(), "No GPU — change runtime to T4 GPU"
print(f"\nGPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
print(f"Setup complete. Workdir: {WORK_DIR}")
print(f"Outputs will save to: {OUTPUTS_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning repo...
Copying BERT files from Drive into src/...
Extracting cohen_data.zip...
Installing dependencies...
HF_TOKEN set from Colab Secrets

GPU: Tesla T4 (15.6 GB)
Setup complete. Workdir: /content/Medical-intervention-text-classification
Outputs will save to: /content/drive/MyDrive/cohen_bert_run/outputs


## 2. ConfigurationEdit `EMAIL` to your address before the first run. Other values can stay as defaults.

In [ ]:
TOPIC = "Statins"
EMAIL = "sammy.okmens@gmail.com"  # your NCBI Entrez email — required by usage policy

# Multi-seed scope for H-Pub2 sensitivity analysis (Statins only).
# Five seeds is the size suggested in CU 167 §5 and paper_draft_v1.md
# reviewer note 2. Order doesn't matter; this set is just spread across the
# small-integer space.
SEEDS = [42, 7, 13, 21, 31]

# Modes that define the H-Pub2 gap (expert vs mechanical MeSH assignment).
# These are the modes the seed loop runs over.
H_PUB2_MODES = ["title_abstract_mesh", "auto_mesh"]

# Baseline modes — not part of the multi-seed run, kept available
# in cells 4a/4b in case the baseline ever needs re-running. Single seed=42
# results from prior runs already cover these in paper_draft_v1.md.
BASELINE_MODES = ["abstract", "title_abstract"]

# Hyperparameters (identical to the original H-Pub2 experiment — do not change
# for the multi-seed run, since the point is to vary seed only)
KFOLD = 5
EPOCHS = 3
BATCH_SIZE = 16  # T4 has 15GB — 16 is conservative; 32 also fits
LR = 2e-5
CLASS_WEIGHT = "balanced"
FP16 = True

print("Configuration:")
print(f"  Topic: {TOPIC}")
print(f"  Seeds: {SEEDS}")
print(f"  H-Pub2 modes (multi-seed):  {H_PUB2_MODES}")
print(f"  Baseline modes (single-seed, optional): {BASELINE_MODES}")
print(f"  K-fold: {KFOLD}, Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LR}")
print(f"  Class weight: {CLASS_WEIGHT}, FP16: {FP16}")

Configuration:
  Topic: Statins
  Seeds: [42, 7, 13, 21, 31]
  H-Pub2 modes (multi-seed):  ['title_abstract_mesh', 'auto_mesh']
  Baseline modes (single-seed, optional): ['abstract', 'title_abstract']
  K-fold: 5, Epochs: 3, Batch: 16, LR: 2e-05
  Class weight: balanced, FP16: True


## 3. GPU smoke testQuick verification that BiomedBERT loads and runs on GPU before committing to the full run. Should take ~30 seconds.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_ID = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
print(f"Loading {MODEL_ID}...")

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)
model = model.to("cuda").half()  # FP16

print(f"Model loaded. GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Forward pass on a single statins-like abstract
text = "Atorvastatin 20mg reduced LDL cholesterol by 38% in a randomized trial of 240 patients."
inputs = tok(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
with torch.no_grad():
    out = model(**inputs)
print(f"Forward pass: logits shape {out.logits.shape}, values {out.logits[0].tolist()}")

# Free up memory before the real run
del model, tok
torch.cuda.empty_cache()
print(f"After cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")
print("GPU verified. Ready for full run.")


Loading microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract...


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/225k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical ar

Model loaded. GPU memory: 0.23 GB
Forward pass: logits shape torch.Size([1, 2]), values [0.0226287841796875, 0.223876953125]
After cleanup: 0.01 GB allocated
GPU verified. Ready for full run.


## 4. Run the experiment

The two H-Pub2-critical modes (cells 4c and 4d) run as **seed loops**: each cell trains 5 seeds × 5 folds = 25 models for one mode. The two baseline cells (4a, 4b) are retained as **optional, single-seed** cells in case the baseline ever needs re-running — they are not part of the multi-seed verdict.

**Order:**
1. (Optional) 4a `abstract` — single seed=42
2. (Optional) 4b `title_abstract` — single seed=42
3. **4c `title_abstract_mesh` (expert MeSH)** — five seeds
4. **4d `auto_mesh` (mechanical MeSH)** — five seeds

Each run writes both `bert_statins_<mode>_seed<N>.txt` (human-readable log) and `bert_statins_<mode>_seed<N>.json` (per-fold raw values + run metadata) to `OUTPUTS_DIR`. The summary cell at the end of the notebook reads the JSON files to aggregate across seeds.

**Idempotency:** each loop iteration checks for an existing output file before running, so reruns after a disconnect resume from where they stopped.

### 4a. Mode: abstract (OPTIONAL — single seed baseline)

In [ ]:
# OPTIONAL baseline run. Not part of multi-seed H-Pub2 scope.
# Skip unless explicitly redoing the baseline.
import subprocess, sys, time
mode = "abstract"
seed = 42
output_file = OUTPUTS_DIR / f"bert_{TOPIC.lower()}_{mode}_seed{seed}.txt"

if output_file.exists():
    print(f"[skip] {output_file.name} already exists; remove to re-run")
else:
    t0 = time.time()
    cmd = [
        sys.executable, "-m", "src.cohen_bert_pipeline",
        "--topic", TOPIC, "--email", EMAIL,
        "--text-mode", mode,
        "--kfold", str(KFOLD), "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE), "--learning-rate", str(LR),
        "--class-weight", CLASS_WEIGHT,
        "--fp16",
        "--seed", str(seed),
        "--output-file", str(output_file),
    ]
    print(f"Running: {mode} seed={seed}")
    subprocess.run(cmd, cwd=WORK_DIR, check=False)
    elapsed = time.time() - t0
    print(f"\n{mode} seed={seed} complete in {elapsed/60:.1f} min")
    print(f"Output: {output_file}")
    print(f"JSON:   {output_file.with_suffix('.json')}")

Running: abstract seed=42

abstract seed=42 complete in 15.2 min
Output: /content/drive/MyDrive/cohen_bert_run/outputs/bert_statins_abstract_seed42.txt
JSON:   /content/drive/MyDrive/cohen_bert_run/outputs/bert_statins_abstract_seed42.json


### 4b. Mode: title_abstract (OPTIONAL — single seed baseline)

In [ ]:
# OPTIONAL baseline run. Not part of multi-seed H-Pub2 scope.
import subprocess, sys, time
mode = "title_abstract"
seed = 42
output_file = OUTPUTS_DIR / f"bert_{TOPIC.lower()}_{mode}_seed{seed}.txt"

if output_file.exists():
    print(f"[skip] {output_file.name} already exists; remove to re-run")
else:
    t0 = time.time()
    cmd = [
        sys.executable, "-m", "src.cohen_bert_pipeline",
        "--topic", TOPIC, "--email", EMAIL,
        "--text-mode", mode,
        "--kfold", str(KFOLD), "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE), "--learning-rate", str(LR),
        "--class-weight", CLASS_WEIGHT,
        "--fp16",
        "--seed", str(seed),
        "--output-file", str(output_file),
    ]
    print(f"Running: {mode} seed={seed}")
    subprocess.run(cmd, cwd=WORK_DIR, check=False)
    elapsed = time.time() - t0
    print(f"\n{mode} seed={seed} complete in {elapsed/60:.1f} min")
    print(f"Output: {output_file}")
    print(f"JSON:   {output_file.with_suffix('.json')}")

Running: title_abstract seed=42

title_abstract seed=42 complete in 15.2 min
Output: /content/drive/MyDrive/cohen_bert_run/outputs/bert_statins_title_abstract_seed42.txt
JSON:   /content/drive/MyDrive/cohen_bert_run/outputs/bert_statins_title_abstract_seed42.json


### 4c. Mode: title_abstract_mesh (expert MeSH) — multi-seed

Trains 5 seeds × 5 folds = 25 models. Expected runtime ~2.5 hours T4. Cell is idempotent: existing seed outputs are skipped, so reruns after a disconnect resume from the unfinished seed.

In [ ]:
import subprocess, sys, time
mode = "title_abstract_mesh"

for seed in SEEDS:
    output_file = OUTPUTS_DIR / f"bert_{TOPIC.lower()}_{mode}_seed{seed}.txt"
    if output_file.exists():
        print(f"[skip] {output_file.name} already exists; remove to re-run")
        continue

    t0 = time.time()
    cmd = [
        sys.executable, "-m", "src.cohen_bert_pipeline",
        "--topic", TOPIC, "--email", EMAIL,
        "--text-mode", mode,
        "--kfold", str(KFOLD), "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE), "--learning-rate", str(LR),
        "--class-weight", CLASS_WEIGHT,
        "--fp16",
        "--seed", str(seed),
        "--output-file", str(output_file),
    ]
    print(f"Running: {mode} seed={seed}")
    subprocess.run(cmd, cwd=WORK_DIR, check=False)
    elapsed = time.time() - t0
    print(f"  {mode} seed={seed} complete in {elapsed/60:.1f} min")
    print(f"  Output: {output_file}")
    print(f"  JSON:   {output_file.with_suffix('.json')}")
    print()

print(f"All {len(SEEDS)} seeds for {mode} complete.")

Running: title_abstract_mesh seed=42
  title_abstract_mesh seed=42 complete in 15.4 min
  Output: /content/drive/MyDrive/cohen_bert_run/outputs/bert_statins_title_abstract_mesh_seed42.txt
  JSON:   /content/drive/MyDrive/cohen_bert_run/outputs/bert_statins_title_abstract_mesh_seed42.json

Running: title_abstract_mesh seed=7
  title_abstract_mesh seed=7 complete in 15.4 min
  Output: /content/drive/MyDrive/cohen_bert_run/outputs/bert_statins_title_abstract_mesh_seed7.txt
  JSON:   /content/drive/MyDrive/cohen_bert_run/outputs/bert_statins_title_abstract_mesh_seed7.json

Running: title_abstract_mesh seed=13
  title_abstract_mesh seed=13 complete in 15.3 min
  Output: /content/drive/MyDrive/cohen_bert_run/outputs/bert_statins_title_abstract_mesh_seed13.txt
  JSON:   /content/drive/MyDrive/cohen_bert_run/outputs/bert_statins_title_abstract_mesh_seed13.json

Running: title_abstract_mesh seed=21
  title_abstract_mesh seed=21 complete in 15.3 min
  Output: /content/drive/MyDrive/cohen_bert_ru

### 4d. Mode: auto_mesh (mechanical MeSH lookup) — multi-seed

Trains 5 seeds × 5 folds = 25 models. Expected runtime ~2.5 hours T4. Cell is idempotent.

In [ ]:
import subprocess, sys, time
mode = "auto_mesh"

for seed in SEEDS:
    output_file = OUTPUTS_DIR / f"bert_{TOPIC.lower()}_{mode}_seed{seed}.txt"
    if output_file.exists():
        print(f"[skip] {output_file.name} already exists; remove to re-run")
        continue

    t0 = time.time()
    cmd = [
        sys.executable, "-m", "src.cohen_bert_pipeline",
        "--topic", TOPIC, "--email", EMAIL,
        "--text-mode", mode,
        "--kfold", str(KFOLD), "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE), "--learning-rate", str(LR),
        "--class-weight", CLASS_WEIGHT,
        "--fp16",
        "--seed", str(seed),
        "--output-file", str(output_file),
    ]
    print(f"Running: {mode} seed={seed}")
    subprocess.run(cmd, cwd=WORK_DIR, check=False)
    elapsed = time.time() - t0
    print(f"  {mode} seed={seed} complete in {elapsed/60:.1f} min")
    print(f"  Output: {output_file}")
    print(f"  JSON:   {output_file.with_suffix('.json')}")
    print()

print(f"All {len(SEEDS)} seeds for {mode} complete.")

Running: auto_mesh seed=42
  auto_mesh seed=42 complete in 15.8 min
  Output: /content/drive/MyDrive/cohen_bert_run/outputs/bert_statins_auto_mesh_seed42.txt
  JSON:   /content/drive/MyDrive/cohen_bert_run/outputs/bert_statins_auto_mesh_seed42.json

Running: auto_mesh seed=7
  auto_mesh seed=7 complete in 15.8 min
  Output: /content/drive/MyDrive/cohen_bert_run/outputs/bert_statins_auto_mesh_seed7.txt
  JSON:   /content/drive/MyDrive/cohen_bert_run/outputs/bert_statins_auto_mesh_seed7.json

Running: auto_mesh seed=13
  auto_mesh seed=13 complete in 15.8 min
  Output: /content/drive/MyDrive/cohen_bert_run/outputs/bert_statins_auto_mesh_seed13.txt
  JSON:   /content/drive/MyDrive/cohen_bert_run/outputs/bert_statins_auto_mesh_seed13.json

Running: auto_mesh seed=21
  auto_mesh seed=21 complete in 15.8 min
  Output: /content/drive/MyDrive/cohen_bert_run/outputs/bert_statins_auto_mesh_seed21.txt
  JSON:   /content/drive/MyDrive/cohen_bert_run/outputs/bert_statins_auto_mesh_seed21.json

Runn

## 5. H-Pub2 multi-seed summary

Reads the per-seed JSON files for the two H-Pub2 modes and reports:

1. Per-seed mean expert MeSH WSS@95, auto MeSH WSS@95, and the expert−auto gap.
2. Pooled mean and standard deviation across seeds (and across all 25 fold values per mode).
3. The seed-sensitivity verdict: is the gap stable in sign across all five seeds? In magnitude?
4. Comparison to the BoW reference gap of 0.121 and to the original single-seed (seed=42) BERT result.

In [ ]:
import json
import statistics
from pathlib import Path

# Modes that define the H-Pub2 gap
EXPERT_MODE = "title_abstract_mesh"
AUTO_MODE = "auto_mesh"

def load_seed_runs(topic, mode, seeds, outputs_dir):
    runs = {}
    for seed in seeds:
        json_path = outputs_dir / f"bert_{topic.lower()}_{mode}_seed{seed}.json"
        if not json_path.exists():
            continue
        payload = json.loads(json_path.read_text(encoding="utf-8"))
        runs[seed] = payload["result"]
    return runs

expert_runs = load_seed_runs(TOPIC, EXPERT_MODE, SEEDS, OUTPUTS_DIR)
auto_runs = load_seed_runs(TOPIC, AUTO_MODE, SEEDS, OUTPUTS_DIR)

print("=" * 92)
print(f"H-Pub2 Multi-Seed Sensitivity — Cohen {TOPIC} (BiomedBERT)")
print("=" * 92)

# Per-seed table
print(f"\n{'Seed':<8} {'Expert WSS mean':>17} {'Auto WSS mean':>15} {'Gap (expert-auto)':>20}")
print("-" * 92)
per_seed_gaps = []
for seed in SEEDS:
    if seed not in expert_runs or seed not in auto_runs:
        print(f"{seed:<8} (incomplete)")
        continue
    exp_mean = expert_runs[seed]["wss_mean"]
    auto_mean = auto_runs[seed]["wss_mean"]
    gap = exp_mean - auto_mean
    per_seed_gaps.append(gap)
    print(f"{seed:<8} {exp_mean:>17.3f} {auto_mean:>15.3f} {gap:>20.3f}")

if len(per_seed_gaps) < 2:
    print("\nNot enough completed seeds for statistics. Run more seeds.")
else:
    # Pooled across all 25 fold values per mode (5 seeds x 5 folds)
    expert_all_folds = [f["wss_at_95"] for r in expert_runs.values() for f in r["folds"]
                        if f["wss_at_95"] is not None]
    auto_all_folds = [f["wss_at_95"] for r in auto_runs.values() for f in r["folds"]
                      if f["wss_at_95"] is not None]

    print(f"\n{'Pooled across seeds':<32}")
    print("-" * 92)
    print(f"Expert MeSH WSS@95 pooled mean: {statistics.mean(expert_all_folds):.3f}  "
          f"std: {statistics.stdev(expert_all_folds):.3f}  (n={len(expert_all_folds)})")
    print(f"Auto MeSH   WSS@95 pooled mean: {statistics.mean(auto_all_folds):.3f}  "
          f"std: {statistics.stdev(auto_all_folds):.3f}  (n={len(auto_all_folds)})")
    print(f"Gap (expert-auto), pooled:      {statistics.mean(expert_all_folds) - statistics.mean(auto_all_folds):+.3f}")

    print(f"\nPer-seed gap mean: {statistics.mean(per_seed_gaps):+.3f}  "
          f"std: {statistics.stdev(per_seed_gaps):.3f}  (n={len(per_seed_gaps)})")
    print(f"Per-seed gap range: [{min(per_seed_gaps):+.3f}, {max(per_seed_gaps):+.3f}]")

    # Sign-stability verdict
    pos = sum(1 for g in per_seed_gaps if g > 0.02)
    neg = sum(1 for g in per_seed_gaps if g < -0.02)
    near_zero = len(per_seed_gaps) - pos - neg

    print(f"\nSign stability: {pos} positive (gap > +0.02), {neg} negative (gap < -0.02), "
          f"{near_zero} near zero (|gap| <= 0.02)")
    print(f"\nFor reference: BoW logistic regression expert-vs-auto gap = +0.121")
    print(f"               Original single-seed (seed=42) BERT result is in paper_draft_v1.md")

# Save consolidated multi-seed result to a single JSON
out_summary = {
    "topic": TOPIC,
    "seeds_run": list(expert_runs.keys()),
    "expert_mode": EXPERT_MODE,
    "auto_mode": AUTO_MODE,
    "per_seed_gaps": {str(s): expert_runs[s]["wss_mean"] - auto_runs[s]["wss_mean"]
                      for s in expert_runs if s in auto_runs},
    "expert_runs": {str(s): expert_runs[s] for s in expert_runs},
    "auto_runs": {str(s): auto_runs[s] for s in auto_runs},
}
summary_path = OUTPUTS_DIR / f"bert_{TOPIC.lower()}_multiseed_summary.json"
summary_path.write_text(json.dumps(out_summary, indent=2, default=str))
print(f"\nConsolidated multi-seed summary saved to: {summary_path}")

H-Pub2 Multi-Seed Sensitivity — Cohen Statins (BiomedBERT)

Seed       Expert WSS mean   Auto WSS mean    Gap (expert-auto)
--------------------------------------------------------------------------------------------
42                   0.275           0.273                0.002
7                    0.256           0.253                0.004
13                   0.185           0.144                0.040
21                   0.342           0.282                0.060
31                   0.307           0.314               -0.007

Pooled across seeds             
--------------------------------------------------------------------------------------------
Expert MeSH WSS@95 pooled mean: 0.273  std: 0.130  (n=25)
Auto MeSH   WSS@95 pooled mean: 0.253  std: 0.130  (n=25)
Gap (expert-auto), pooled:      +0.020

Per-seed gap mean: +0.020  std: 0.029  (n=5)
Per-seed gap range: [-0.007, +0.060]

Sign stability: 2 positive (gap > +0.02), 0 negative (gap < -0.02), 3 near zero (|gap| <= 0.02)



## 6. Backup — bundle outputs for downloadOptional cell. Creates a single zip of all outputs that you can download from Drive in one click. The individual `.txt` files are already in Drive, so this just makes archival easier.

In [ ]:
import shutil
from pathlib import Path
import datetime

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
zip_path = DRIVE_ROOT / f"bert_{TOPIC.lower()}_multiseed_{stamp}.zip"

shutil.make_archive(str(zip_path.with_suffix("")), "zip", str(OUTPUTS_DIR))
print(f"Outputs bundled to: {zip_path}")
print(f"Size: {zip_path.stat().st_size / 1024:.1f} KB")
print(f"\nAll output files in: {OUTPUTS_DIR}")
for fp in sorted(OUTPUTS_DIR.iterdir()):
    print(f"  {fp.name} ({fp.stat().st_size / 1024:.1f} KB)")

Outputs bundled to: /content/drive/MyDrive/cohen_bert_run/bert_statins_multiseed_20260618_1635.zip
Size: 44.5 KB

All output files in: /content/drive/MyDrive/cohen_bert_run/outputs
  bert_adhd_abstract.json (2.2 KB)
  bert_adhd_abstract.txt (2.2 KB)
  bert_adhd_auto_mesh.json (2.2 KB)
  bert_adhd_auto_mesh.txt (2.2 KB)
  bert_adhd_results.json (0.6 KB)
  bert_adhd_title_abstract.json (2.2 KB)
  bert_adhd_title_abstract.txt (2.2 KB)
  bert_adhd_title_abstract_mesh.json (2.2 KB)
  bert_adhd_title_abstract_mesh.txt (2.2 KB)
  bert_opiods_abstract.txt (3.5 KB)
  bert_opiods_auto_mesh.txt (3.5 KB)
  bert_opiods_results.json (0.6 KB)
  bert_opiods_title_abstract.txt (3.5 KB)
  bert_opiods_title_abstract_mesh.txt (3.5 KB)
  bert_statins_abstract.txt (4.8 KB)
  bert_statins_abstract_seed42.json (2.2 KB)
  bert_statins_abstract_seed42.txt (4.8 KB)
  bert_statins_auto_mesh.txt (4.8 KB)
  bert_statins_auto_mesh_seed13.json (2.2 KB)
  bert_statins_auto_mesh_seed13.txt (4.8 KB)
  bert_statins_auto_